In [316]:
import yfinance as yf
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np


silver = yf.download("SI=F", start = "2011-01-01")['Close']
gold = yf.download("GC=F", start = "2011-01-01")['Close']


data = pd.concat([silver, gold], axis = 1) #align indexes
data.columns = ['Silver','Gold']
data['GSR'] = data['Gold'] / data['Silver']
data = data.reset_index() 
data = data.dropna()
data



/var/folders/pp/8s7gqcrn3rscx70q717ydwzm0000gn/T/ipykernel_92592/971171764.py:8: FutureWarning: YF.download() has changed argument auto_adjust default to True
  silver = yf.download("SI=F", start = "2011-01-01")['Close']
[*********************100%***********************]  1 of 1 completed
/var/folders/pp/8s7gqcrn3rscx70q717ydwzm0000gn/T/ipykernel_92592/971171764.py:9: FutureWarning: YF.download() has changed argument auto_adjust default to True
  gold = yf.download("GC=F", start = "2011-01-01")['Close']
[*********************100%***********************]  1 of 1 completed


,Date,Silver,Gold,GSR
0,2011-01-03,31.096001,1422.599976,45.748648
1,2011-01-04,29.492001,1378.500000,46.741488
2,2011-01-05,29.173000,1373.400024,47.077778
3,2011-01-06,29.110001,1371.400024,47.110958
4,2011-01-07,28.660999,1368.500000,47.747812
...,...,...,...,...
3924,2026-08-13,64.873001,4363.600098,67.263731
3925,2026-08-14,64.987999,4380.399902,67.403212
3926,2026-08-17,66.121002,4417.799805,66.813866
3927,2026-08-18,63.941002,4366.000000,68.281695


In [317]:
## Stage 1: Prediction Feature Definitions

data['Silver_Return_1D'] = data['Silver'].pct_change()
data['Silver_Return_1D']

data['Silver_Return_5D_Forward'] = (
    data['Silver'].shift(-5) / data['Silver'] - 1
)

data['Silver_Return_5D_Forward'] 
data['Silver_Momentum_20D'] = data['Silver'].pct_change(20)
data['Silver_Volatility_20D'] = (
    data['Silver_Return_1D'].rolling(20).std()
)
data['Gold_Momentum_20D'] = data['Gold'].pct_change(20)

model_data = data.dropna(
    subset=['Silver_Return_5D_Forward', 'Silver_Momentum_20D',
    'Silver_Volatility_20D']
).copy()


model_data




,Date,Silver,Gold,GSR,Silver_Return_1D,Silver_Return_5D_Forward,Silver_Momentum_20D,Silver_Volatility_20D,Gold_Momentum_20D
20,2011-02-01,28.524000,1339.599976,46.963959,0.012423,0.061247,-0.082712,0.021483,-0.058344
21,2011-02-02,28.299000,1331.500000,47.051133,-0.007888,0.069755,-0.040452,0.018399,-0.034095
22,2011-02-03,28.733000,1352.300049,47.064353,0.015336,0.047263,-0.015082,0.018660,-0.015363
23,2011-02-04,29.063999,1348.300049,46.390727,0.011520,0.031930,-0.001580,0.018849,-0.016844
24,2011-02-07,29.348000,1347.599976,45.917950,0.009772,0.040241,0.023970,0.018598,-0.015272
...,...,...,...,...,...,...,...,...,...
3919,2026-08-06,61.438999,4242.000000,69.044093,-0.010628,0.055893,0.017573,0.023728,0.026969
3920,2026-08-07,63.332001,4340.700195,68.538814,0.030811,0.026148,0.058904,0.024479,0.057650
3921,2026-08-10,65.106003,4361.799805,66.995356,0.028011,0.015590,0.129646,0.023209,0.091268
3922,2026-08-11,64.768997,4383.000000,67.671265,-0.005176,-0.012784,0.102038,0.023122,0.079264


In [318]:
## Stage 2: Train-Validation-Test 
train = model_data[
    model_data['Date'] < '2021-01-01'
].copy()

validation = model_data[
    (model_data['Date'] >= '2021-01-01') &
    (model_data['Date'] < '2024-01-01')
].copy()

test = model_data[
    model_data['Date'] >= '2024-01-01'
].copy()




In [319]:
#Centering GSR to avoid multicollinearity (added due to high VIF for gsr and gsr^2)
gsr_mean = train['GSR'].mean()

train['GSR_centered'] = train['GSR'] - gsr_mean
validation['GSR_centered'] = validation['GSR'] - gsr_mean

train['GSR_centered^2'] = train['GSR_centered'] ** 2
validation['GSR_centered^2'] = validation['GSR_centered'] ** 2

In [320]:
#GSR only test
from sklearn.linear_model import LinearRegression

X_train_gsr_only = train[['GSR_centered']]

gsr_only_model = LinearRegression()
gsr_only_model.fit(
    X_train_gsr_only,
    train['Silver_Return_5D_Forward']
)

X_val_gsr_only = validation[['GSR_centered']]
y_val = validation['Silver_Return_5D_Forward']
gsr_only_pred = gsr_only_model.predict(X_val_gsr_only)

print("MAE:", mean_absolute_error(y_val, gsr_only_pred))
print(
    "RMSE:",
    np.sqrt(mean_squared_error(y_val, gsr_only_pred))
)
print("R^2:", r2_score(y_val, gsr_only_pred))
print(
    "Correlation:",
    np.corrcoef(y_val, gsr_only_pred)[0, 1]
)
print(
    "Directional Accuracy:",
    (np.sign(y_val) == np.sign(gsr_only_pred)).mean()
)

MAE: 0.03137007772211383
RMSE: 0.04017941931809548
R^2: 0.005038810905333646
Correlation: 0.14996432833313572
Directional Accuracy: 0.5298804780876494


In [321]:
#GSR plus feature/s on Train data
from sklearn.linear_model import LinearRegression
X_train = train[['GSR_centered', 'Silver_Momentum_20D','Silver_Volatility_20D']]
y_train = train['Silver_Return_5D_Forward']
gsr_model = LinearRegression()
gsr_model.fit(X_train, y_train)
print(f"Intercept: {gsr_model.intercept_}")
print(f"GSR coeff: {gsr_model.coef_[0]}")
print(f"20-day silver momentum: {gsr_model.coef_[1]}")
print(f"20-day silver vol: {gsr_model.coef_[2]}")
#print(f"20-day Gold Momentum vol: {gsr_model.coef_[3]}")

Intercept: -0.005037038972019899
GSR coeff: 0.0003365991699946408
20-day silver momentum: -0.013937419152543696
20-day silver vol: 0.33536899367740125


In [322]:
#GSR baseline plus feats on Val Set
X_val = validation[['GSR_centered','Silver_Momentum_20D', 'Silver_Volatility_20D']]
y_val = validation['Silver_Return_5D_Forward']

validation['GSR_Prediction'] = gsr_model.predict(X_val)

validation[
    ['Date', 'GSR', 'Silver_Return_5D_Forward', 'GSR_Prediction']
]

,Date,GSR,Silver_Return_5D_Forward,GSR_Prediction
2514,2021-01-04,71.276203,-0.074952,-0.000064
2515,2021-01-05,70.824417,-0.079322,-0.000300
2516,2021-01-06,70.696625,-0.053869,0.000547
2517,2021-01-07,70.305147,-0.053456,0.000236
2518,2021-01-08,74.611501,0.009885,0.005847
...,...,...,...,...
3262,2023-12-22,84.689173,-0.022931,0.006579
3263,2023-12-26,85.246853,-0.049619,0.006965
3264,2023-12-27,85.411278,-0.056862,0.007023
3265,2023-12-28,85.932702,-0.041933,0.007421


In [323]:
#GSR Baseline Eval Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
pred_val = validation['GSR_Prediction']
mae = mean_absolute_error(y_val, pred_val)
rmse = np.sqrt(mean_squared_error(y_val, pred_val))
r2 = r2_score(y_val, pred_val)

correlation = np.corrcoef(y_val, pred_val)[0, 1]
directional_accuracy = (
    np.sign(y_val) == np.sign(pred_val)
).mean()

print("MAE:", mae)
print("RMSE:", rmse)
print("R^2:", r2)
print("Prediction/Actual Correlation:", correlation)
print("Directional Accuracy:", directional_accuracy)

MAE: 0.03119865098733041
RMSE: 0.039980897046844904
R^2: 0.014846517977270102
Prediction/Actual Correlation: 0.21271464243183733
Directional Accuracy: 0.50199203187251


MAE: 0.03119865098733041
RMSE: 0.039980897046844904
R^2: 0.014846517977270102
Prediction/Actual Correlation: 0.21271464243183733
Directional Accuracy: 0.50199203187251

GSR exhibits weak out-of-sample predictive information, but is insufficient as a standalone forecasting model.


In [324]:
# GSR Degree 2 baseline model
train['GSR^2']= train['GSR']**2
validation['GSR^2']= validation['GSR']**2

X_train_quad = train[['GSR','GSR^2']]
y_train_quad = train['Silver_Return_5D_Forward']

quad_model = LinearRegression()
quad_model.fit(X_train_quad, y_train_quad)

print("Intercept:", quad_model.intercept_)
print("GSR coefficient:", quad_model.coef_[0])

print("GSR^2 coefficient:", quad_model.coef_[1])


X_val_quad = validation[['GSR', 'GSR^2']]
validation['Quad_Prediction'] = quad_model.predict(X_val_quad)
quad_pred = validation['Quad_Prediction']


quad_mae = mean_absolute_error(y_val, quad_pred)
quad_rmse = np.sqrt(mean_squared_error(y_val, quad_pred))
quad_r2 = r2_score(y_val, quad_pred)

quad_corr = np.corrcoef(y_val, quad_pred)[0, 1]

quad_direction = (
    np.sign(y_val) == np.sign(quad_pred)
).mean()

print("Quadratic Model")
print("MAE:", quad_mae)
print("RMSE:", quad_rmse)
print("R^2:", quad_r2)
print("Correlation:", quad_corr)
print("Directional Accuracy:", quad_direction)

Intercept: 0.05295373444235778
GSR coefficient: -0.0018712311628622145
GSR^2 coefficient: 1.540234027304203e-05
Quadratic Model
MAE: 0.03133347935702811
RMSE: 0.039884437128322806
R^2: 0.019594444970869818
Correlation: 0.16712638470442465
Directional Accuracy: 0.5179282868525896


In [325]:
#GSR Degree 2 plus feat/s training

train['GSR_centered^2'] = train['GSR_centered'] ** 2
validation['GSR_centered^2'] = validation['GSR_centered'] ** 2

X_train_quad = train[['GSR_centered','GSR_centered^2','Silver_Momentum_20D','Silver_Volatility_20D']]
y_train_quad = train['Silver_Return_5D_Forward']

quad_model = LinearRegression()
quad_model.fit(X_train_quad, y_train_quad)

print("Intercept:", quad_model.intercept_)
print("GSR coefficient:", quad_model.coef_[0])
print("GSR^2 coefficient:", quad_model.coef_[1])
print("20-day silver momentum coefficient:", quad_model.coef_[2])
print("20-day silver vol coefficient:", quad_model.coef_[3])


Intercept: -0.004047548412673482
GSR coefficient: 0.0003068388532672398
GSR^2 coefficient: 1.4878142058199782e-05
20-day silver momentum coefficient: -0.021553056625334042
20-day silver vol coefficient: 0.10353295031237855


In [326]:
#GSR degree 2 model plus feat/s evaluation

X_val_quad = validation[['GSR_centered', 'GSR_centered^2','Silver_Momentum_20D','Silver_Volatility_20D']]
validation['Quad_Prediction'] = quad_model.predict(X_val_quad)
quad_pred = validation['Quad_Prediction']


quad_mae = mean_absolute_error(y_val, quad_pred)
quad_rmse = np.sqrt(mean_squared_error(y_val, quad_pred))
quad_r2 = r2_score(y_val, quad_pred)

quad_corr = np.corrcoef(y_val, quad_pred)[0, 1]

quad_direction = (
    np.sign(y_val) == np.sign(quad_pred)
).mean()

print("Quadratic Model")
print("MAE:", quad_mae)
print("RMSE:", quad_rmse)
print("R^2:", quad_r2)
print("Correlation:", quad_corr)
print("Directional Accuracy:", quad_direction)

Quadratic Model
MAE: 0.03119537992648684
RMSE: 0.03972396514237051
R^2: 0.02746774793892992
Correlation: 0.19624042314268456
Directional Accuracy: 0.547144754316069


In [327]:
# Post model fitting, exploring predictor relationships
features = [
    'GSR_centered',
    'GSR_centered^2',
    'Silver_Momentum_20D',
    'Silver_Volatility_20D'
]
corr_matrix = train[features].corr()
corr_matrix

,GSR_centered,GSR_centered^2,Silver_Momentum_20D,Silver_Volatility_20D
GSR_centered,1.000000,-0.002902,-0.053224,-0.208090
GSR_centered^2,-0.002902,1.000000,0.083042,0.372665
Silver_Momentum_20D,-0.053224,0.083042,1.000000,-0.126273
Silver_Volatility_20D,-0.208090,0.372665,-0.126273,1.000000


In [328]:
#VIF to check for multicollinearity
from statsmodels.stats.outliers_influence import variance_inflation_factor

X_vif = train[features].copy()

vif_data = pd.DataFrame()
vif_data['Feature'] = X_vif.columns
vif_data['VIF'] = [
    variance_inflation_factor(X_vif.values, i)
    for i in range(X_vif.shape[1])
]

vif_data

,Feature,VIF
0,GSR_centered,1.017528
1,GSR_centered^2,1.677306
2,Silver_Momentum_20D,1.022958
3,Silver_Volatility_20D,1.680031


In [329]:
#Testing consistency of coeffs

early_train = train[train['Date'] < '2013-01-01'].copy()
mid_train = train[
     (train['Date'] >= '2013-01-01') &
     (train['Date'] < '2016-01-01')
     ]
late_train = train[train['Date'] >= '2016-01-01'].copy()

early_model = LinearRegression()
early_model.fit(
    early_train[features],
    early_train['Silver_Return_5D_Forward']
)

mid_model = LinearRegression()
mid_model.fit(
    mid_train[features],
    mid_train['Silver_Return_5D_Forward']
)

late_model = LinearRegression()
late_model.fit(
    late_train[features],
    late_train['Silver_Return_5D_Forward']
)

     

print(f"Pre 2013 model: {early_model.coef_ }")
print(f"2013-2016 model: {mid_model.coef_ }")
print(f"Post 2016 model: {late_model.coef_ }")
     
     







Pre 2013 model: [-2.06183430e-03 -4.25946751e-05 -4.34740448e-02 -7.18610233e-01]
2013-2016 model: [ 6.80789391e-04  3.70283802e-06 -5.83443444e-02  4.79603510e-01]
Post 2016 model: [ 2.11046421e-04  1.64653601e-05 -1.73319006e-02  1.81356758e-01]


In [330]:
#Rolling out of window year-by-year backtest

results = []

for year in range(2014, 2021):

    expanding_train = train[
        train['Date'] < f'{year}-01-01'
    ].copy()

    next_year = train[
        (train['Date'] >= f'{year}-01-01') &
        (train['Date'] < f'{year + 1}-01-01')
    ].copy()

    expanding_gsr_mean = expanding_train['GSR'].mean()
    
    
    expanding_train['GSR_centered'] = (
        expanding_train['GSR'] - expanding_gsr_mean
    )

    expanding_train['GSR_centered^2'] = (
        expanding_train['GSR_centered'] ** 2
    )


    next_year['GSR_centered'] = (
        next_year['GSR'] - expanding_gsr_mean
    )

    next_year['GSR_centered^2'] = (
        next_year['GSR_centered'] ** 2
    )

    features = [
        'GSR_centered',
        'GSR_centered^2',
        'Silver_Momentum_20D',
        'Silver_Volatility_20D'
    ]

    model = LinearRegression()

    model.fit(
        expanding_train[features],
        expanding_train['Silver_Return_5D_Forward']
    )

    predictions = model.predict(
        next_year[features]
    )

    y_true = next_year['Silver_Return_5D_Forward']

    results.append({
        'Year': year,
        'MAE': mean_absolute_error(y_true, predictions),
        'RMSE': np.sqrt(
            mean_squared_error(y_true, predictions)
        ),
        'R2': r2_score(y_true, predictions),
        'Correlation': np.corrcoef(
            y_true,
            predictions
        )[0, 1],
        'Directional_Accuracy': (
            np.sign(y_true) == np.sign(predictions)
        ).mean()
    })

expanding_results = pd.DataFrame(results)

expanding_results


,Year,MAE,RMSE,R2,Correlation,Directional_Accuracy
0,2014,0.022739,0.029728,-0.001304,0.085873,0.619048
1,2015,0.026636,0.035153,0.014044,0.301895,0.583333
2,2016,0.030070,0.039058,0.002425,0.218982,0.588000
3,2017,0.022440,0.026842,0.031685,0.189430,0.553785
4,2018,0.018602,0.023465,-0.205760,0.355578,0.476000
5,2019,0.019498,0.025768,0.026366,0.346213,0.527778
6,2020,0.045004,0.069018,0.039030,0.207986,0.577075


In [331]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge


alphas = [0.01, 0.1, 1, 10, 100]

ridge_results = []

for alpha in alphas:

    ridge_model = make_pipeline(
        StandardScaler(),
        Ridge(alpha=alpha)
    )

    ridge_model.fit(X_train, y_train)

    ridge_pred = ridge_model.predict(X_val)

    ridge_results.append({
        'Alpha': alpha,
        'MAE': mean_absolute_error(y_val, ridge_pred),
        'RMSE': np.sqrt(mean_squared_error(y_val, ridge_pred)),
        'R2': r2_score(y_val, ridge_pred),
        'Correlation': np.corrcoef(y_val, ridge_pred)[0, 1],
        'Directional_Accuracy': (
            np.sign(y_val) == np.sign(ridge_pred)
        ).mean()
    })

ridge_results_df = pd.DataFrame(ridge_results)

ridge_results_df

,Alpha,MAE,RMSE,R2,Correlation,Directional_Accuracy
0,0.01,0.031199,0.039981,0.014847,0.212715,0.501992
1,0.10,0.031199,0.039981,0.014846,0.212715,0.501992
2,1.00,0.031199,0.039981,0.014846,0.212720,0.501992
3,10.00,0.031199,0.039981,0.014842,0.212764,0.501992
4,100.00,0.031198,0.039982,0.014773,0.213168,0.503320


After centering the polynomial GSR terms and reducing multicollinearity, Ridge regularization did not materially improve out-of-sample validation performance relative to OLS. Therefore, the simpler OLS specification was retained.

In [ ]:
#Model on testing data 2024-2026
train_val = pd.concat([train, validation]).copy()
final_gsr_mean = train_val['GSR'].mean()

train_val['GSR_centered'] = (
    train_val['GSR'] - final_gsr_mean
)

train_val['GSR_centered^2'] = (
    train_val['GSR_centered'] ** 2
)

test['GSR_centered'] = (
    test['GSR'] - final_gsr_mean
)

test['GSR_centered^2'] = (
    test['GSR_centered'] ** 2
)

final_model = LinearRegression()
final_model.fit(
    train_val[features], train_val['Silver_Return_5D_Forward']
    )

test_pred = final_model.predict(
    test[features]
)
y_test = test['Silver_Return_5D_Forward']




In [280]:
#Evaluation Metrics for Model on Testing Period
print(
    "MAE:",
    mean_absolute_error(y_test, test_pred)
)

print(
    "RMSE:",
    np.sqrt(
        mean_squared_error(y_test, test_pred)
    )
)

print(
    "R^2:",
    r2_score(y_test, test_pred)
)

print(
    "Correlation:",
    np.corrcoef(y_test, test_pred)[0, 1]
)

print(
    "Directional Accuracy:",
    (
        np.sign(y_test) ==
        np.sign(test_pred)
    ).mean()
)


MAE: 0.04350259748666823
RMSE: 0.06242581555330967
R^2: -0.0009039073907508577
Correlation: 0.08639605428876476
Directional Accuracy: 0.5966514459665144


The model did not survive as a strong magnitude forecaster, but it may have survived as a directional trading signal

In [336]:
#Directional strategy with 5-day pred return, plus sharpe


strategy = test.copy()
strategy["Prediction"] = test_pred
strategy['Signal'] = np.sign(strategy['Prediction'])


strategy_5d = strategy.iloc[::5].copy()
strategy_5d

strategy_5d['Strategy_Return'] = (
    strategy_5d['Signal'] *
    strategy_5d['Silver_Return_5D_Forward']
)
strategy_5d['Strategy_Return']
strategy_5d['Cumulative_Wealth'] = (
    1 + strategy_5d['Strategy_Return']
).cumprod()


total_return = (
    strategy_5d['Cumulative_Wealth'].iloc[-1] - 1
)

periods_per_year = 252 / 5

sharpe = (
    strategy_5d['Strategy_Return'].mean()
    /
    strategy_5d['Strategy_Return'].std()
) * np.sqrt(periods_per_year)

print("Sharpe:", sharpe)



Sharpe: 0.6901133420048449


In [338]:
# Strategy setup

strategy = test.copy()

# Final model predictions already created earlier
strategy['Prediction'] = test_pred

# +1 = long, -1 = short
strategy['Signal'] = np.sign(strategy['Prediction'])

# Use non-overlapping 5-day trading periods
strategy_5d = strategy.iloc[::5].copy()

# Number of 5-day periods in a trading year
periods_per_year = 252 / 5


In [343]:
#Adjusting Vol Weight

strategy_5d['Annualized_Silver_Vol'] = (
    strategy_5d['Silver_Volatility_20D'] * np.sqrt(252)
)
strategy_5d['Annualized_Silver_Vol']

target_vol = 0.15

strategy_5d['Vol_Weight'] = (
    target_vol /
    strategy_5d['Annualized_Silver_Vol']
)

strategy_5d['Vol_Weight'] = (
    strategy_5d['Vol_Weight'].clip(upper=1.0)
)

strategy_5d['Vol_Target_Position'] = (
    strategy_5d['Signal'] *
    strategy_5d['Vol_Weight']
)
strategy_5d['Vol_Target_Return'] = (
    strategy_5d['Vol_Target_Position'] *
    strategy_5d['Silver_Return_5D_Forward']
)


In [342]:
# Metrics

periods_per_year = 252 / 5

strategy_5d['Vol_Target_Wealth'] = (
    1 + strategy_5d['Vol_Target_Return']
).cumprod()

vol_target_sharpe = (
    strategy_5d['Vol_Target_Return'].mean()
    /
    strategy_5d['Vol_Target_Return'].std()
) * np.sqrt(periods_per_year)

n_periods = len(strategy_5d)

vol_target_annual_return = (
    strategy_5d['Vol_Target_Wealth'].iloc[-1]
    ** (periods_per_year / n_periods)
    - 1
)

vol_target_annual_vol = (
    strategy_5d['Vol_Target_Return'].std()
    * np.sqrt(periods_per_year)
)

running_peak = strategy_5d['Vol_Target_Wealth'].cummax()

vol_target_drawdown = (
    strategy_5d['Vol_Target_Wealth']
    / running_peak
) - 1

vol_target_max_drawdown = vol_target_drawdown.min()

print("Vol Target Sharpe:", vol_target_sharpe)
print("Vol Target Annualized Return:", vol_target_annual_return)
print("Vol Target Annualized Vol:", vol_target_annual_vol)
print("Vol Target Max Drawdown:", vol_target_max_drawdown)

Vol Target Sharpe: 1.17789249162385
Vol Target Annualized Return: 0.18530448382404274
Vol Target Annualized Vol: 0.15456997255926155
Vol Target Max Drawdown: -0.18111247273218045


return: 21.5% → 18.5%
volatility: 39.5% → 15.5%
drawdown: -54.8% → -18.1%
Sharpe: 0.68 → 1.18